In [2]:
!pip install chemprop rdkit -q

In [7]:
import pandas as pd
df=pd.read_csv('/kaggle/input/datasets/amirtesh/alzheimer-targets/AD_MTDL_combined_chembl36.csv')
df.head()

,smiles,AChE_class,BuChE_class,BACE1_class,MAO_B_class
0,Br.CCOC(=O)C(C(=O)N/N=C/c1ccc([N+](=O)[O-])cc1...,NaN,NaN,NaN,Active
1,Br.CC[N+](CC)(CCCCCn1c(=O)cc(C)n(CCCCCCn2c(C)c...,Active,Moderate,NaN,NaN
2,Br.CC[N+](CC)(CCCCCn1c(=O)cc(C)n(CCCCCn2c(C)cc...,Active,Inactive,NaN,NaN
3,Br.CC[N+](CC)(CCCCCn1c(=O)cc(C)n(CCCCn2c(C)cc(...,Active,Inactive,NaN,NaN
4,Br.CC[N+](CC)(CCCCCn1c(=O)cc(C)n(CCCn2c(C)cc(=...,Active,Moderate,NaN,NaN


In [ ]:
import numpy as np
import pandas as pd

class_mapping = {"Active": 1, "Inactive": 0, "Moderate": np.nan}

target_cols = ["AChE_class", "BuChE_class", "BACE1_class", "MAO_B_class"]

for col in target_cols:
    df[col] = df[col].map(class_mapping)

df = df.dropna(subset=target_cols, how="all").reset_index(drop=True)

# Inspect your new clean binary dataset
print(f"Total compounds ready for modeling: {len(df):,}")
print("\nNew class distributions (excluding NaN):")
for col in target_cols:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))

Total compounds ready for modeling: 11,297

New class distributions (excluding NaN):

AChE_class:
AChE_class
NaN    8597
1.0    1536
0.0    1164
Name: count, dtype: int64

BuChE_class:
BuChE_class
NaN    9597
1.0     939
0.0     761
Name: count, dtype: int64

BACE1_class:
BACE1_class
NaN    5808
1.0    4739
0.0     750
Name: count, dtype: int64

MAO_B_class:
MAO_B_class
NaN    9184
0.0    1135
1.0     978
Name: count, dtype: int64


In [9]:
df.head()

,smiles,AChE_class,BuChE_class,BACE1_class,MAO_B_class
0,Br.CCOC(=O)C(C(=O)N/N=C/c1ccc([N+](=O)[O-])cc1...,NaN,NaN,NaN,1.0
1,Br.CC[N+](CC)(CCCCCn1c(=O)cc(C)n(CCCCCCn2c(C)c...,1.0,NaN,NaN,NaN
2,Br.CC[N+](CC)(CCCCCn1c(=O)cc(C)n(CCCCCn2c(C)cc...,1.0,0.0,NaN,NaN
3,Br.CC[N+](CC)(CCCCCn1c(=O)cc(C)n(CCCCn2c(C)cc(...,1.0,0.0,NaN,NaN
4,Br.CC[N+](CC)(CCCCCn1c(=O)cc(C)n(CCCn2c(C)cc(=...,1.0,NaN,NaN,NaN


In [ ]:
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning.pytorch as pl
from rdkit import Chem
from sklearn.metrics import (
    roc_auc_score, average_precision_score, matthews_corrcoef,
    f1_score, confusion_matrix
)
from chemprop import data, featurizers, models
from chemprop import nn as cpnn
from chemprop.nn import BCELoss as ChempropBCELoss 

RANDOM_SEED  = 42
DATA_PATH    = "/kaggle/input/datasets/amirtesh/alzheimer-targets/AD_MTDL_combined_chembl36.csv"
CKPT_DIR     = "/kaggle/working/checkpoints"
TARGET_COLS  = ["AChE_class", "BuChE_class", "BACE1_class", "MAO_B_class"]

pl.seed_everything(RANDOM_SEED)


df = pd.read_csv(DATA_PATH)

class_mapping = {"Active": 1, "Inactive": 0, "Moderate": np.nan}
for col in TARGET_COLS:
    df[col] = df[col].map(class_mapping)

df = df.dropna(subset=TARGET_COLS, how="all").reset_index(drop=True)

valid_mask = df["smiles"].apply(lambda s: Chem.MolFromSmiles(str(s)) is not None)
df = df[valid_mask].reset_index(drop=True)

print(f"Compounds after cleaning: {len(df):,}")
for col in TARGET_COLS:
    n_pos = (df[col] == 1).sum()
    n_neg = (df[col] == 0).sum()
    n_nan = df[col].isna().sum()
    print(f"  {col:<15}: Active={n_pos:>5}  Inactive={n_neg:>5}  NaN={n_nan:>5}")

targets_array = df[TARGET_COLS].to_numpy(dtype=np.float32)

datapoints = [
    data.MoleculeDatapoint.from_smi(smi, y)
    for smi, y in zip(df["smiles"].tolist(), targets_array)
]

mols = [d.mol for d in datapoints]
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    train_idx, val_idx, test_idx = data.make_split_indices(
        mols, split="scaffold_balanced", sizes=(0.8, 0.1, 0.1)
    )

train_data, val_data, test_data = data.split_data_by_indices(
    datapoints, train_idx, val_idx, test_idx
)
train_data = train_data[0]
val_data   = val_data[0]
test_data  = test_data[0]

print(f"\nSplit — train: {len(train_data):,}  val: {len(val_data):,}  test: {len(test_data):,}")

featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()

train_dset = data.MoleculeDataset(train_data, featurizer)
val_dset   = data.MoleculeDataset(val_data,   featurizer)
test_dset  = data.MoleculeDataset(test_data,  featurizer)

train_loader = data.build_dataloader(train_dset, shuffle=True,  num_workers=2)
val_loader   = data.build_dataloader(val_dset,   shuffle=False, num_workers=2)
test_loader  = data.build_dataloader(test_dset,  shuffle=False, num_workers=2)

train_targets = np.array([d.y for d in train_data], dtype=np.float32)

pos_weights = []
print("\nClass imbalance (train split):")
for t, col in enumerate(TARGET_COLS):
    vals  = train_targets[:, t]
    valid = vals[~np.isnan(vals)]
    n_pos = (valid == 1).sum()
    n_neg = (valid == 0).sum()
    # raw ratio, clipped to reasonable range
    pw = float(np.clip(n_neg / max(n_pos, 1), 0.2, 8.0))
    pos_weights.append(pw)
    print(f"  {col:<15}: n_pos={n_pos:>5}  n_neg={n_neg:>5}  pos_weight={pw:.3f}")

pos_weight_tensor = torch.tensor(pos_weights, dtype=torch.float32)


class WeightedBCELoss(ChempropBCELoss):
    def __init__(self, pos_weight: torch.Tensor):
        super().__init__()
        self.register_buffer("pos_weight", pos_weight)

    def _calc_unreduced_loss(
        self, preds: torch.Tensor, targets: torch.Tensor, *args, **kwargs
    ) -> torch.Tensor:
        return F.binary_cross_entropy_with_logits(
            preds,
            targets,
            reduction="none",
            pos_weight=self.pos_weight.to(preds.device),
        )

criterion = WeightedBCELoss(pos_weight_tensor)

mpnn_block = cpnn.BondMessagePassing()
pooling    = cpnn.MeanAggregation()

ffn = cpnn.BinaryClassificationFFN(
    input_dim  = mpnn_block.output_dim,
    n_tasks    = len(TARGET_COLS),
    hidden_dim = 300,
    n_layers   = 2,
    dropout    = 0.1,
    criterion  = criterion,
)

model = models.MPNN(
    message_passing = mpnn_block,
    agg             = pooling,
    predictor       = ffn,
    batch_norm      = True,
    metrics         = [cpnn.metrics.BinaryAUPRC(), cpnn.metrics.BinaryAUROC()],
)

print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

checkpoint_cb = pl.callbacks.ModelCheckpoint(
    monitor    = "val/prc",
    mode       = "max",
    save_top_k = 1,
    dirpath    = CKPT_DIR,
    filename   = "best-{epoch:02d}-{val/prc:.3f}",
)

early_stop_cb = pl.callbacks.EarlyStopping(
    monitor   = "val/prc",
    patience  = 15,
    mode      = "max",
)

trainer = pl.Trainer(
    max_epochs          = 75,
    accelerator         = "gpu" if torch.cuda.is_available() else "cpu",
    devices             = 1,
    callbacks           = [checkpoint_cb, early_stop_cb],
    enable_checkpointing= True,
    log_every_n_steps   = 10,
    enable_progress_bar = True,
    logger              = False,
)

trainer.fit(model, train_loader, val_loader)
print(f"\nBest checkpoint: {checkpoint_cb.best_model_path}")

Seed set to 42


Compounds after cleaning: 11,297
  AChE_class     : Active= 1536  Inactive= 1164  NaN= 8597
  BuChE_class    : Active=  939  Inactive=  761  NaN= 9597
  BACE1_class    : Active= 4739  Inactive=  750  NaN= 5808
  MAO_B_class    : Active=  978  Inactive= 1135  NaN= 9184


The return type of make_split_indices has changed in v2.1 - see help(make_split_indices)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.



Split — train: 9,039  val: 1,129  test: 1,129

Class imbalance (train split):
  AChE_class     : n_pos= 1238  n_neg=  960  pos_weight=0.775
  BuChE_class    : n_pos=  782  n_neg=  623  pos_weight=0.797
  BACE1_class    : n_pos= 3786  n_neg=  618  pos_weight=0.200
  MAO_B_class    : n_pos=  750  n_neg=  874  pos_weight=1.165

Total parameters: 410,104


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing      │  227 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation         │      0 │ train │     0 │
│ 2 │ bn              │ BatchNorm1d             │    600 │ train │     0 │
│ 3 │ predictor       │ BinaryClassificationFFN │  181 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity                │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList              │      0 │ train │     0 │
└───┴─────────────────┴─────────────────────────┴────────┴───────┴───────┘

Trainable params: 410 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 410 K                                                                                                
Total estimated model params size (MB): 1.640                                                                      
Modules in train mode: 27                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()


Best checkpoint: /kaggle/working/checkpoints/best-epoch=36-val/prc=0.978.ckpt


In [ ]:
import torch.serialization
torch.serialization.add_safe_globals([
    WeightedBCELoss,
    cpnn.metrics.BinaryAUPRC,
    cpnn.metrics.BinaryAUROC,
])

best_model = models.MPNN.load_from_checkpoint(checkpoint_cb.best_model_path)
best_model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
best_model = best_model.to(device)

def get_preds_and_targets(loader):
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in loader:
            bmg, X_vd, features, targets, weights, lt_mask, gt_mask = batch
            bmg.V          = bmg.V.to(device)
            bmg.E          = bmg.E.to(device)
            bmg.edge_index = bmg.edge_index.to(device)
            bmg.batch      = bmg.batch.to(device)
            X_vd     = X_vd.to(device) if X_vd is not None else None
            features = [f.to(device) for f in features] if features else features
            logits   = best_model(bmg, X_vd, features)
            probs    = torch.sigmoid(logits)
            all_preds.append(probs.cpu().numpy())
            all_targets.append(targets.numpy())
    return np.vstack(all_preds), np.vstack(all_targets)

val_preds,  val_targets  = get_preds_and_targets(val_loader)
test_preds, test_targets = get_preds_and_targets(test_loader)

thresholds_to_try = np.round(np.arange(0.05, 0.95, 0.01), 2)
optimal_thresholds = {}

print("\n===== THRESHOLD OPTIMIZATION (val set) =====\n")
for t, col in enumerate(TARGET_COLS):
    y_true = val_targets[:, t]
    y_prob = val_preds[:, t]
    mask   = ~np.isnan(y_true)
    y_true_m = y_true[mask].astype(int)
    y_prob_m = y_prob[mask]

    best_mcc, best_thresh = -1.0, 0.5
    for thresh in thresholds_to_try:
        y_pred = (y_prob_m >= thresh).astype(int)
        if len(np.unique(y_pred)) < 2:
            continue
        mcc = matthews_corrcoef(y_true_m, y_pred)
        if mcc > best_mcc:
            best_mcc    = mcc
            best_thresh = thresh

    optimal_thresholds[col] = best_thresh
    print(f"  {col:<15}: threshold = {best_thresh:.2f}   val MCC = {best_mcc:.4f}")

rows = []
for t, col in enumerate(TARGET_COLS):
    y_true = test_targets[:, t]
    y_prob = test_preds[:, t]
    mask   = ~np.isnan(y_true)
    y_true = y_true[mask].astype(int)
    y_prob = y_prob[mask]

    thresh = optimal_thresholds[col]
    y_pred = (y_prob >= thresh).astype(int)

    auroc  = roc_auc_score(y_true, y_prob)
    auprc  = average_precision_score(y_true, y_prob)
    mcc    = matthews_corrcoef(y_true, y_pred)
    f1_mac = f1_score(y_true, y_pred, average="macro",    zero_division=0)
    f1_w   = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    if len(np.unique(y_pred)) == 2:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        sens = tp / max(tp + fn, 1)  # recall on positives
        spec = tn / max(tn + fp, 1)  # recall on negatives
    else:
        tn = fp = fn = tp = 0
        sens = spec = 0.0

    rows.append({
        "Task":      col,
        "N":         int(mask.sum()),
        "Threshold": round(thresh, 2),
        "AUROC":     round(auroc,  4),
        "AUPRC":     round(auprc,  4),
        "MCC":       round(mcc,    4),
        "F1_Mac":    round(f1_mac, 4),
        "F1_W":      round(f1_w,   4),
        "Sens":      round(sens,   4),
        "Spec":      round(spec,   4),
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
    })

results_df = pd.DataFrame(rows).set_index("Task")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:.4f}".format)

print("\n===== TEST METRICS (optimal thresholds) =====\n")
print(results_df[["N","Threshold","AUROC","AUPRC","MCC","F1_Mac","F1_W","Sens","Spec"]].to_string())

print("\n===== CONFUSION MATRICES =====")
for _, r in results_df.iterrows():
    print(f"\n  {r.name}  (threshold={r.Threshold})")
    print(f"             Pred 0   Pred 1")
    print(f"  Actual 0   {int(r.TN):>6}   {int(r.FP):>6}")
    print(f"  Actual 1   {int(r.FN):>6}   {int(r.TP):>6}")


===== THRESHOLD OPTIMIZATION (val set) =====

  AChE_class     : threshold = 0.59   val MCC = 0.8063
  BuChE_class    : threshold = 0.68   val MCC = 0.9182
  BACE1_class    : threshold = 0.60   val MCC = 0.8441
  MAO_B_class    : threshold = 0.51   val MCC = 0.6154

===== TEST METRICS (optimal thresholds) =====

               N  Threshold  AUROC  AUPRC    MCC  F1_Mac   F1_W   Sens   Spec
Task                                                                         
AChE_class   306     0.5900 0.9281 0.9630 0.7202  0.8584 0.8674 0.8615 0.8739
BuChE_class  173     0.6800 0.9651 0.9602 0.8837  0.9413 0.9420 0.9684 0.9103
BACE1_class  531     0.6000 0.9856 0.9983 0.8066  0.9032 0.9626 0.9768 0.8421
MAO_B_class  187     0.5100 0.8532 0.8770 0.5354  0.7645 0.7643 0.7113 0.8222

===== CONFUSION MATRICES =====

  AChE_class  (threshold=0.59)
             Pred 0   Pred 1
  Actual 0       97       14
  Actual 1       27      168

  BuChE_class  (threshold=0.68)
             Pred 0   Pred 1
  Ac

In [ ]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score, matthews_corrcoef,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)

def evaluate(preds, targets, thresholds, split_name):
    rows = []
    for t, col in enumerate(TARGET_COLS):
        y_true = targets[:, t]
        y_prob = preds[:, t]
        mask   = ~np.isnan(y_true)
        y_true = y_true[mask].astype(int)
        y_prob = y_prob[mask]

        thresh = thresholds[col]
        y_pred = (y_prob >= thresh).astype(int)

        auroc  = roc_auc_score(y_true, y_prob)
        auprc  = average_precision_score(y_true, y_prob)
        mcc    = matthews_corrcoef(y_true, y_pred)
        acc    = accuracy_score(y_true, y_pred)

        prec_w = precision_score(y_true, y_pred, average="weighted", zero_division=0)
        rec_w  = recall_score(y_true, y_pred,    average="weighted", zero_division=0)
        f1_w   = f1_score(y_true, y_pred,        average="weighted", zero_division=0)

        prec_m = precision_score(y_true, y_pred, average="macro",    zero_division=0)
        rec_m  = recall_score(y_true, y_pred,    average="macro",    zero_division=0)
        f1_m   = f1_score(y_true, y_pred,        average="macro",    zero_division=0)

        prec_1 = precision_score(y_true, y_pred, average="binary",   zero_division=0)
        rec_1  = recall_score(y_true, y_pred,    average="binary",   zero_division=0)
        f1_1   = f1_score(y_true, y_pred,        average="binary",   zero_division=0)

        if len(np.unique(y_pred)) == 2:
            tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        else:
            tn = fp = fn = tp = 0

        rows.append({
            "Task"      : col,
            "Split"     : split_name,
            "N"         : int(mask.sum()),
            "Threshold" : round(thresh,  2),
            "AUROC"     : round(auroc,   4),
            "AUPRC"     : round(auprc,   4),
            "MCC"       : round(mcc,     4),
            "Accuracy"  : round(acc,     4),
            "Prec_W"    : round(prec_w,  4),
            "Rec_W"     : round(rec_w,   4),
            "F1_W"      : round(f1_w,    4),
            "Prec_Mac"  : round(prec_m,  4),
            "Rec_Mac"   : round(rec_m,   4),
            "F1_Mac"    : round(f1_m,    4),
            "Prec_Pos"  : round(prec_1,  4),
            "Rec_Pos"   : round(rec_1,   4),
            "F1_Pos"    : round(f1_1,    4),
            "TP": int(tp), "FP": int(fp),
            "TN": int(tn), "FN": int(fn),
        })
    return pd.DataFrame(rows).set_index(["Task", "Split"])


pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)

val_results  = evaluate(val_preds,  val_targets,  optimal_thresholds, "val")
test_results = evaluate(test_preds, test_targets, optimal_thresholds, "test")

all_results = pd.concat([val_results, test_results]).sort_index(level="Task")

metric_cols = [
    "N", "Threshold",
    "AUROC", "AUPRC", "MCC", "Accuracy",
    "Prec_W",   "Rec_W",   "F1_W",
    "Prec_Mac", "Rec_Mac", "F1_Mac",
    "Prec_Pos", "Rec_Pos", "F1_Pos",
]

print("\n===== FULL METRICS — VAL + TEST =====\n")
print(all_results[metric_cols].to_string())

print("\n===== CONFUSION MATRICES =====")
for (task, split), r in all_results.iterrows():
    print(f"\n  {task}  [{split}]  (threshold={r.Threshold})")
    print(f"               Pred 0   Pred 1")
    print(f"  Actual 0     {int(r.TN):>6}   {int(r.FP):>6}")
    print(f"  Actual 1     {int(r.FN):>6}   {int(r.TP):>6}")


===== FULL METRICS — VAL + TEST =====

                     N  Threshold  AUROC  AUPRC    MCC  Accuracy  Prec_W  Rec_W   F1_W  Prec_Mac  Rec_Mac  F1_Mac  Prec_Pos  Rec_Pos  F1_Pos
Task        Split                                                                                                                           
AChE_class  test   306     0.5900 0.9281 0.9630 0.7202    0.8660  0.8720 0.8660 0.8674    0.8527   0.8677  0.8584    0.9231   0.8615  0.8912
            val    196     0.5900 0.9353 0.9366 0.8063    0.9031  0.9036 0.9031 0.9031    0.9027   0.9036  0.9029    0.9200   0.8932  0.9064
BACE1_class test   531     0.6000 0.9856 0.9983 0.8066    0.9623  0.9630 0.9623 0.9626    0.8972   0.9094  0.9032    0.9809   0.9768  0.9789
            val    554     0.6000 0.9838 0.9972 0.8441    0.9639  0.9635 0.9639 0.9637    0.9269   0.9173  0.9220    0.9771   0.9812  0.9792
BuChE_class test   173     0.6800 0.9651 0.9602 0.8837    0.9422  0.9429 0.9422 0.9420    0.9444   0.9393  0.9413 

In [ ]:
import os
import json
import shutil
from datetime import datetime


SAVE_DIR = "/kaggle/working/AD_MTDL_model"
os.makedirs(SAVE_DIR, exist_ok=True)

ckpt_dest = os.path.join(SAVE_DIR, "AD_MTDL_best.ckpt")
shutil.copy2(checkpoint_cb.best_model_path, ckpt_dest)
print(f"Checkpoint saved: {ckpt_dest}")

def collect_metrics(preds, targets, thresholds, split_name):
    split_metrics = {}
    for t, col in enumerate(TARGET_COLS):
        y_true = targets[:, t]
        y_prob = preds[:, t]
        mask   = ~np.isnan(y_true)
        y_true_m = y_true[mask].astype(int)
        y_prob_m = y_prob[mask]
        thresh   = thresholds[col]
        y_pred   = (y_prob_m >= thresh).astype(int)

        tn, fp, fn, tp = (0, 0, 0, 0)
        if len(np.unique(y_pred)) == 2:
            tn, fp, fn, tp = confusion_matrix(y_true_m, y_pred).ravel()

        split_metrics[col] = {
            "n_compounds"  : int(mask.sum()),
            "n_active"     : int(y_true_m.sum()),
            "n_inactive"   : int((y_true_m == 0).sum()),
            "threshold"    : float(thresh),
            "AUROC"        : round(float(roc_auc_score(y_true_m, y_prob_m)),          4),
            "AUPRC"        : round(float(average_precision_score(y_true_m, y_prob_m)),4),
            "MCC"          : round(float(matthews_corrcoef(y_true_m, y_pred)),         4),
            "Accuracy"     : round(float(accuracy_score(y_true_m, y_pred)),            4),
            "F1_macro"     : round(float(f1_score(y_true_m, y_pred, average="macro",    zero_division=0)), 4),
            "F1_weighted"  : round(float(f1_score(y_true_m, y_pred, average="weighted", zero_division=0)), 4),
            "Precision_pos": round(float(precision_score(y_true_m, y_pred, average="binary", zero_division=0)), 4),
            "Recall_pos"   : round(float(recall_score(y_true_m, y_pred,    average="binary", zero_division=0)), 4),
            "confusion_matrix": {
                "TN": int(tn), "FP": int(fp),
                "FN": int(fn), "TP": int(tp),
            },
        }
    return split_metrics

metadata = {
    "project"      : "AD-MTDL Multi-task ChemProp Classifier",
    "created"      : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "chemprop_version": chemprop.__version__,
    "random_seed"  : RANDOM_SEED,
    "targets"      : TARGET_COLS,
    "label_scheme" : {
        "Active"  : 1,
        "Inactive": 0,
        "Moderate": "excluded (NaN — not used in training)",
    },
    "data_filters" : {
        "source"            : "ChEMBL 36",
        "assay_type"        : ["B", "F"],
        "confidence_score"  : ">=8",
        "standard_relation" : "=",
        "pchembl_not_null"  : True,
        "potential_duplicate": 0,
        "deduplication"     : "median pChEMBL per unique compound",
        "active_threshold"  : "pChEMBL >= 7.0",
        "inactive_threshold": "pChEMBL < 5.0",
        "moderate_excluded" : "5.0 <= pChEMBL < 7.0",
    },
    "split": {
        "method"    : "scaffold_balanced",
        "sizes"     : [0.8, 0.1, 0.1],
        "train_n"   : len(train_data),
        "val_n"     : len(val_data),
        "test_n"    : len(test_data),
    },
    "architecture" : {
        "model"       : "MPNN (ChemProp v2)",
        "message_passing": "BondMessagePassing",
        "aggregation" : "MeanAggregation",
        "ffn_hidden"  : 300,
        "ffn_layers"  : 2,
        "dropout"     : 0.1,
        "batch_norm"  : True,
        "n_tasks"     : len(TARGET_COLS),
        "total_params": sum(p.numel() for p in best_model.parameters()),
    },
    "training" : {
        "loss"           : "WeightedBCELoss (BCEWithLogitsLoss + per-task pos_weight)",
        "pos_weights"    : {col: round(pos_weights[t], 4) for t, col in enumerate(TARGET_COLS)},
        "max_epochs"     : 75,
        "early_stopping" : "patience=15, monitor=val/prc",
        "best_epoch"     : int(checkpoint_cb.best_model_path.split("epoch=")[1].split("-")[0]),
        "best_val_prc"   : float(checkpoint_cb.best_model_score),
    },
    "threshold_optimization" : {
        "method"  : "Youden-MCC sweep on val set (threshold in [0.05, 0.95], step=0.01)",
        "metric"  : "MCC maximization",
    },
    "optimal_thresholds" : {
        col: float(optimal_thresholds[col]) for col in TARGET_COLS
    },
    "metrics" : {
        "val" : collect_metrics(val_preds,  val_targets,  optimal_thresholds, "val"),
        "test": collect_metrics(test_preds, test_targets, optimal_thresholds, "test"),
    },
}

json_path = os.path.join(SAVE_DIR, "AD_MTDL_metadata.json")
with open(json_path, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"Metadata saved: {json_path}")


csv_path = os.path.join(SAVE_DIR, "AD_MTDL_results.csv")
all_results.reset_index().to_csv(csv_path, index=False)
print(f"Results CSV saved: {csv_path}")

zip_path = "/kaggle/working/AD_MTDL_model"
shutil.make_archive(zip_path, "zip", SAVE_DIR)
print(f"\nZip ready for download: {zip_path}.zip")
print("\nContents:")
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(os.path.join(SAVE_DIR, f))
    print(f"  {f:<40} {size/1024:.1f} KB")

Checkpoint saved: /kaggle/working/AD_MTDL_model/AD_MTDL_best.ckpt
Metadata saved: /kaggle/working/AD_MTDL_model/AD_MTDL_metadata.json
Results CSV saved: /kaggle/working/AD_MTDL_model/AD_MTDL_results.csv

Zip ready for download: /kaggle/working/AD_MTDL_model.zip

Contents:
  AD_MTDL_metadata.json                    5.6 KB
  AD_MTDL_best.ckpt                        4834.8 KB
  AD_MTDL_results.csv                      1.1 KB
